# CatBoost 超参调优（v1 保持 + GPU 兼容修复）

In [ ]:

# ================= 配置区 =================
REF_DATE_STR = "2025-08-31"

TRAIN_CSV = "train.csv"
TEST_CSV  = "testaa.csv"
TRAIN_STM_FEAT = "train_statement_feature.csv"
TEST_STM_FEAT  = "testaa_statement_feature.csv"

OUT_DIR   = "outputs"
N_FOLDS   = 5
RANDOM_STATE = 1337

# 搜索预算
N_TRIALS  = 30

# 是否使用 GPU（True/False）
USE_GPU = True
# ========================================


In [ ]:

import os, gc, json, math, random
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

REF_DATE = pd.Timestamp(REF_DATE_STR)

def _to_days_since_now(unix_series):
    dt = pd.to_datetime(unix_series, unit='s', utc=True, errors='coerce').dt.tz_convert(None)
    return (REF_DATE - dt).dt.days

def prepare_main_table(df):
    use_cols = [
        'id','title','career','zip_code','residence','loan','term','interest_rate',
        'issue_time','syndicated','installment','record_time','history_time',
        'total_accounts','balance_accounts','balance_limit','balance','level'
    ] + (['label'] if 'label' in df.columns else [])
    df = df[use_cols].copy()
    for c in ['issue_time','record_time','history_time']:
        df[f'{c}_days'] = _to_days_since_now(df[c])
    df['diff_issue_record_days']   = df['issue_time_days'] - df['record_time_days']
    df['diff_issue_history_days']  = df['issue_time_days'] - df['history_time_days']
    df['diff_record_history_days'] = df['record_time_days'] - df['history_time_days']
    df['utilization']    = (df['balance'] / df['balance_limit']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 10)
    df['accounts_ratio'] = (df['balance_accounts'] / df['total_accounts']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)
    def split_level(x):
        if isinstance(x, str) and len(x) >= 2: return x[0], x[1:]
        return 'NA', 'NA'
    lv = df['level'].fillna('NA')
    df['grade'], df['subgrade'] = zip(*lv.map(split_level))
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    for c in cat_cols:
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = df[c].astype('Int64').astype(str)
        else:
            df[c] = df[c].astype(str)
        df[c] = df[c].fillna('NA')
    return df

def merge_statement_feats(main_df, stm_path):
    if os.path.exists(stm_path):
        stm = pd.read_csv(stm_path)
        stm = stm[[c for c in stm.columns if c != 'label']].copy()
        main_df = main_df.merge(stm, on='id', how='left')
    else:
        main_df['stm_missing'] = 1
    num_exclude = ['id','label','title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    stm_num_cols = [c for c in main_df.columns if c not in num_exclude]
    main_df[stm_num_cols] = main_df[stm_num_cols].fillna(0)
    return main_df

def get_feature_lists(df):
    drop_cols = ['id','label']
    features = [c for c in df.columns if c not in drop_cols]
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    cat_cols = [c for c in cat_cols if c in features]
    return features, cat_cols


In [ ]:

def sanitize_params(params, use_gpu=False, y=None, seed=None):
    p = dict(params)
    if use_gpu and 'rsm' in p:
        p.pop('rsm', None)
    if p.get('bootstrap_type', '').lower() == 'bayesian' and 'subsample' in p:
        p.pop('subsample', None)
        p.setdefault('bagging_temperature', 1.0)
    if use_gpu:
        p['task_type'] = 'GPU'
    if y is not None:
        pos, neg = int(np.sum(y)), int(len(y) - np.sum(y))
        p['scale_pos_weight'] = float(neg / max(pos, 1))
    p.setdefault('loss_function', 'Logloss')
    p.setdefault('eval_metric', 'AUC')
    p.setdefault('verbose', False)
    if seed is not None:
        p['random_seed'] = seed
    return p

def pretty_params(p):
    rsm = p.get('rsm', None)
    base = f"depth={p['depth']}, lr={p['learning_rate']:.4f}, l2={p['l2_leaf_reg']:.2f}, subsample={p['subsample']:.2f}"
    if rsm is not None:
        base += f", rsm={rsm:.2f}"
    base += f", rs={p['random_strength']:.2f}, ohms={p['one_hot_max_size']}"
    return base


In [ ]:

def build_folds(y, n_folds=5, seed=1337):
    return StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

def evaluate_params(df_train, features, cat_cols, params, n_folds=5, seed=1337):
    X = df_train[features]
    y = df_train['label'].astype(int).values
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]

    p = sanitize_params(params, use_gpu=USE_GPU, y=y, seed=seed)

    folds = build_folds(y, n_folds=n_folds, seed=seed)
    oof = np.zeros(len(y), dtype=float)
    fold_scores, models = [], []
    fi_accum = np.zeros(len(features), dtype=float)

    for f, (tr_idx, va_idx) in enumerate(folds.split(X, y), 1):
        train_pool = Pool(X.iloc[tr_idx], label=y[tr_idx], cat_features=cat_idx)
        valid_pool = Pool(X.iloc[va_idx], label=y[va_idx], cat_features=cat_idx)
        model = CatBoostClassifier(**p)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=300)
        pred = model.predict_proba(valid_pool)[:,1]
        oof[va_idx] = pred
        fold_scores.append(roc_auc_score(y[va_idx], pred))
        fi_accum += np.array(model.get_feature_importance(train_pool, type='FeatureImportance'))
        models.append(model)

    mean_auc = roc_auc_score(y, oof)
    fi_mean = pd.DataFrame({'feature': X.columns.tolist(), 'importance': fi_accum/len(fold_scores)}).sort_values('importance', ascending=False)
    return mean_auc, fold_scores, oof, models, fi_mean


In [ ]:

# 读取数据保持 v1 风格
tr = pd.read_csv(TRAIN_CSV)
tr = prepare_main_table(tr)
tr = merge_statement_feats(tr, TRAIN_STM_FEAT)
features, cat_cols = get_feature_lists(tr)

print("训练样本维度：", tr.shape)
print("特征数：", len(features))
print("类别特征：", cat_cols)
print("label 分布：", tr['label'].value_counts(normalize=True).round(4).to_dict())


In [ ]:

# 自适应 GPU 的采样空间
def sample_params():
    base = dict(
        iterations = int(np.random.choice([4000, 5000, 6000])),
        learning_rate = float(10 ** np.random.uniform(-1.9, -1.2)),
        depth = int(np.random.choice([5,6,7,8])),
        l2_leaf_reg = float(np.random.uniform(4, 14)),
        bootstrap_type = 'Bernoulli',
        subsample = float(np.random.uniform(0.65, 0.9)),
        random_strength = float(np.random.uniform(1.0, 3.0)),
        one_hot_max_size = int(np.random.choice([5,10,20,30]))
    )
    if not USE_GPU:
        base['rsm'] = float(np.random.uniform(0.6, 0.95))
    return base


In [ ]:

# 随机搜索
results = []
best_auc, best_pack = -1.0, None

for t in range(1, N_TRIALS+1):
    params = sample_params()
    mean_auc, fold_scores, oof, models, fi_mean = evaluate_params(tr, features, cat_cols, params, n_folds=N_FOLDS, seed=RANDOM_STATE + t)

    rec = dict(trial=t, oof_auc=round(mean_auc, 6), folds=str([round(s,6) for s in fold_scores]), **params)
    results.append(rec)
    print(f"[Trial {t:02d}] AUC={mean_auc:.6f} | {pretty_params(params)}")

    if mean_auc > best_auc:
        best_auc = mean_auc
        best_pack = (params, oof, models, fi_mean)

res_df = pd.DataFrame(results).sort_values('oof_auc', ascending=False)
res_path = os.path.join(OUT_DIR, "hpo_results.csv")
res_df.to_csv(res_path, index=False, encoding='utf-8')
print(f"[SAVE] 超参搜索结果 -> {res_path}")
res_df.head(10)


In [ ]:

# 最优 trial 导出 OOF、FI、提交
best_params, best_oof, best_models, best_fi = best_pack
oof_path = os.path.join(OUT_DIR, "oof_predictions_hpo.csv")
fi_path  = os.path.join(OUT_DIR, "feature_importance_hpo.csv")
pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_pred': best_oof}).to_csv(oof_path, index=False, encoding='utf-8')
best_fi.to_csv(fi_path, index=False, encoding='utf-8')
print(f"[SAVE] OOF -> {oof_path} | FI -> {fi_path}")
print(f"[BEST] OOF AUC = {roc_auc_score(tr['label'], best_oof):.6f}")
print("[BEST] Params =", json.dumps(best_params, ensure_ascii=False))

if os.path.exists(TEST_CSV):
    te = pd.read_csv(TEST_CSV)
    te = prepare_main_table(te)
    te = merge_statement_feats(te, TEST_STM_FEAT)
    if 'label' in te.columns:
        te = te.drop(columns=['label'])

    X_te = te[ best_fi['feature'].tolist() ]
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    cat_cols = [c for c in cat_cols if c in X_te.columns]
    pool = Pool(X_te, cat_features=[X_te.columns.get_loc(c) for c in cat_cols])

    preds = np.mean([m.predict_proba(pool)[:,1] for m in best_models], axis=0)
    sub = pd.DataFrame({'id': te['id'], 'prob': preds})
    sub_path = os.path.join(OUT_DIR, 'test_pred_catboost_hpo.csv')
    sub.to_csv(sub_path, index=False, encoding='utf-8')
    print(f"[SAVE] Test predictions -> {sub_path}")
else:
    print("[INFO] 未找到测试集，跳过提交文件生成。")
